# E3 — prueba interactiva de Gradio, DINOv3 y a3_tuned

Notebook preparado para Colab con GPU. Usa el checkpoint `a3_tuned` oficial fijado por etapa 7 y el `official_complete_bundle` de etapa 5. Las imágenes, máscaras y salidas quedan en Drive privado. No abre etiquetas ni calcula métricas. Estado: **preparado, aún no ejecutado con modelos reales**. La demostración manual con `a3_tuned` es exploratoria; ese decoder se entrenó y evaluó con puntos automáticos. La UI se abre mediante el proxy privado de Colab, sin enlace público de Gradio.


In [ ]:
# Requiere Colab con GPU. Acepta Python 3.12 o 3.13; instala dependencias y monta Drive.
import os, sys, subprocess
assert sys.version_info[:2] in {(3, 12), (3, 13)}, f"Usa Colab Python 3.12/3.13; actual: {sys.version}"
subprocess.run(["nvidia-smi", "-L"], check=True)
from google.colab import drive
drive.mount("/content/drive")
os.environ["AUTO_PROMPT_EXECUTION_CONTEXT"] = "google_colab_user_notebook"
os.environ["SAM2_BUILD_CUDA"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install",
    "numpy==2.2.6", "scipy==1.15.3", "pillow==11.3.0", "rasterio==1.4.3",
    "gradio==6.31.0", "transformers==4.57.1", "safetensors==0.6.2",
    "huggingface_hub==0.35.3", "hydra-core==1.3.2", "iopath==0.1.10",
    "tqdm==4.67.1", "setuptools==80.9.0", "wheel==0.45.1"], check=True)
print("Entorno base listo; GPU:", subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip())


In [ ]:
# Obtén el código fuente público desde el commit exacto; no se requieren ZIPs en Drive.
from pathlib import Path
import os, sys, subprocess, shutil
SOURCE_COMMIT = "4f95130"
repo = Path("/content/auto-prompt-sam2-e3-demo")
if repo.exists():
    assert repo.resolve() == Path("/content/auto-prompt-sam2-e3-demo").resolve()
    shutil.rmtree(repo)
subprocess.run(["git", "clone", "--filter=blob:none", "https://github.com/roberto-chavez/auto-prompt-sam2.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", SOURCE_COMMIT], check=True)
manifest = {"commit_HEAD": subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()}
assert manifest["commit_HEAD"].startswith(SOURCE_COMMIT)
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps",
    "git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4"], check=True)
from auto_prompt_sam2.demo_backend import DemoBackend
assert hasattr(DemoBackend, "segment_automatic")
print("Fuentes verificadas; commit:", manifest["commit_HEAD"])


## Cargar el chip y comprobar los checkpoints

El notebook toma el chip Ghana_1033830 desde el Drive del proyecto. El raster y todas las máscaras quedan privados en Drive. El checkpoint `a3_tuned` debe coincidir byte a byte con el pin de `configs/stage7_dependencies.json`.

In [ ]:
from pathlib import Path
import os, json, hashlib
project = Path("/content/drive/MyDrive/auto-prompt-sam2")
chip = project / "raw/v1.1/data/flood_events/HandLabeled/S2Hand/Ghana_1033830_S2Hand.tif"
assert chip.is_file(), f"No encontré el chip: {chip}"
base = project / "regimes/official/stage6/0f8f08fb38e4e459203d37c4d86138a9273ac31f8090d2d1d97278cf98288904/decoder_export/20261101T231354388506Z_175b8b784023/payload"
bundle = project / "stage5/environment/runs/20261029T194653Z_8073c068fa_ed6583db/official_complete_bundle/20261029T202402773328Z_19682f232c35/payload"
assert base.is_file() and bundle.is_file(), "No aparece el checkpoint Stage 6 o bundle Stage 5"
print("chip SHA-256", hashlib.sha256(chip.read_bytes()).hexdigest())
print("a3_tuned bytes/SHA-256", base.stat().st_size, hashlib.sha256(base.read_bytes()).hexdigest())
print("official bundle bytes/SHA-256", bundle.stat().st_size, hashlib.sha256(bundle.read_bytes()).hexdigest())
assert hashlib.sha256(base.read_bytes()).hexdigest() == "ca639740af32dc23b41932d6d3f17426b7b2e2d6d40394ef3fc51b63766bf55a"
os.environ["DEMO_MODE"] = "inference"
os.environ["DEMO_ALLOW_DOWNLOAD"] = "1"
os.environ["DEMO_DECODER_KIND"] = "stage6_a3_tuned"
os.environ["DEMO_DECODER_CHECKPOINT"] = str(base)
os.environ["DEMO_AUTO_BUNDLE"] = str(bundle)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token: os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass
finally:
    if "hf_token" in globals(): del hf_token

In [ ]:
# Inicializa backend con a3_tuned y el bundle automático oficiales.
from auto_prompt_sam2.demo_config import load_demo_config, stage6_decoder_pin
from auto_prompt_sam2.demo_backend import DemoBackend
from auto_prompt_sam2.demo_images import load_image, S2_LAYOUT
import matplotlib.pyplot as plt
cfg = load_demo_config(environ=os.environ)
be = DemoBackend(cfg)
img = load_image(str(chip), S2_LAYOUT, be.params)
plt.figure(figsize=(6,6)); plt.imshow(img.preview()); plt.axis("off"); plt.show()
pin = stage6_decoder_pin()
print("a3_tuned pinned:", pin)
print("RGB:", img.rgb.shape, "válidos:", int(img.valid.sum()))

In [ ]:
# Smoke de backend: manual con comparación a3_tuned, y automático DINOv3→SAM con el mismo decoder.
import torch, json, hashlib
from PIL import Image
from datetime import datetime, timezone
run_dir = project / "demo_gradio_runs" / ("codex_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
run_dir.mkdir(parents=True, exist_ok=False)
POINTS = [(256,256,1),(100,100,0)]
manual = be.segment(img, POINTS, compare=True)
assert manual["frozen"]["mask"].any() and manual["tuned"] is not None, manual["status"]
auto = be.segment_automatic(img, compare=True)
assert auto["frozen"]["mask"].shape == img.valid.shape and auto["tuned"] is not None, auto["status"]
assert auto["automatic_prompts"]["points"] > 0, auto["automatic_prompts"]
for name, mask in [("backend_manual_frozen", manual["frozen"]["mask"]), ("backend_manual_a3_tuned", manual["tuned"]["mask"]), ("backend_auto_frozen", auto["frozen"]["mask"]), ("backend_auto_a3_tuned", auto["tuned"]["mask"])]:
    Image.fromarray(mask.astype("uint8")*255).save(run_dir / f"{name}.png")
smoke = {"created_utc":datetime.now(timezone.utc).isoformat(), "source_commit":manifest.get("commit_HEAD"), "gpu":torch.cuda.get_device_name(0), "chip_sha256":hashlib.sha256(chip.read_bytes()).hexdigest(), "decoder_kind":"stage6_a3_tuned", "decoder_sha256":pin["sha256"], "decoder_generation":pin["generation"], "manual_status":manual["status"], "manual_frozen_pixels":int(manual["frozen"]["mask"].sum()), "manual_tuned_pixels":int(manual["tuned"]["mask"].sum()), "automatic_status":auto["status"], "automatic_prompt_components":auto["automatic_prompts"]["components"], "automatic_prompt_points":auto["automatic_prompts"]["points"], "automatic_threshold":auto["automatic_prompts"]["positive_threshold"], "auto_frozen_pixels":int(auto["frozen"]["mask"].sum()), "auto_tuned_pixels":int(auto["tuned"]["mask"].sum()), "labels_read":False, "metrics_computed":False}
(run_dir / "backend_smoke.json").write_text(json.dumps(smoke,indent=2)+"\n",encoding="utf-8")
print(json.dumps(smoke,indent=2))

## Prueba interactiva privada de Gradio

Se guardan en Drive `results.json`, `gradio.log` y PNG privados. Abre la interfaz desde el enlace del proxy autenticado de Colab (no se genera un enlace público). Prueba Manual y luego Automático · DINOv3 → SAM; en ambas verifica las máscaras y que el detalle registre `a3_tuned`.


In [ ]:
import logging, platform, subprocess, json
from auto_prompt_sam2.demo_ui import DemoController, build_demo
from auto_prompt_sam2.artifacts import sha256_file
run_dir = project / "demo_gradio_runs" / ("ui_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
run_dir.mkdir(parents=True, exist_ok=False)
records_path, log_path = run_dir/"results.json", run_dir/"gradio.log"
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s", handlers=[logging.FileHandler(log_path,encoding="utf-8"),logging.StreamHandler()], force=True)
class EvidenceController(DemoController):
    def segment(self, image, points, compare, strategy="Manual"):
        output = super().segment(image, points, compare, strategy)
        overlay, tuned_overlay, frozen_mask, tuned_mask, details, status = output
        rows = json.loads(records_path.read_text()) if records_path.exists() else []
        saved = {}
        for name, value in [("frozen_mask",frozen_mask),("tuned_mask",tuned_mask)]:
            if value is not None:
                path=run_dir/f"{len(rows)+1:02d}_{name}.png"
                Image.fromarray(__import__("numpy").asarray(value).astype("uint8")).save(path)
                saved[name]={"file":path.name,"sha256":sha256_file(path),"positive_pixels":int((__import__("numpy").asarray(value)>0).sum())}
        record={"index":len(rows)+1,"timestamp_utc":datetime.now(timezone.utc).isoformat(),"source_commit":manifest.get("commit_HEAD"),"python":platform.python_version(),"gpu":torch.cuda.get_device_name(0),"strategy":strategy,"status":status,"details":details,"decoder_kind":"stage6_a3_tuned","decoder_sha256":stage6_decoder_pin()["sha256"],"automatic_bundle_sha256":sha256_file(bundle),"files":saved}
        rows.append(record); records_path.write_text(json.dumps(rows,indent=2,ensure_ascii=False)+"\n",encoding="utf-8")
        logging.getLogger("demo.evidence").info("GRADIO_RESULT %s",json.dumps(record,ensure_ascii=False))
        return output
controller=EvidenceController(cfg)
demo=build_demo(cfg,controller=controller)
server=demo.launch(share=False,server_name="0.0.0.0",server_port=7860,prevent_thread_lock=True,show_error=True)
from google.colab.output import eval_js
private_url=eval_js("google.colab.kernel.proxyPort(7860)")
print("Interfaz privada de Colab:",private_url)
print("Drive run folder:",run_dir)

In [ ]:
# Ejecuta después de terminar las dos rutas en la UI.
rows=json.loads(records_path.read_text()) if records_path.exists() else []
print("Registros UI:",len(rows),"estrategias:",[r["strategy"] for r in rows])
assert any(r["strategy"]=="Manual" and "a3_tuned" in r["status"] for r in rows), "Falta recorrido manual con a3_tuned"
assert any(r["strategy"].startswith("Automático") and r["details"].get("automatic_prompts",{}).get("points",0)>0 and "a3_tuned" in r["status"] for r in rows), "Falta recorrido automático con puntos y a3_tuned"
demo.close()
print("UI cerrada; resultados:",records_path,"log:",log_path,"archivos:",sorted(p.name for p in run_dir.iterdir()))